# Oil Spill - preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/oilspill/00_preprocessing/oilspill_preprocessing.ipynb)

Stateless steps only: label mapping, removing identifiers and duplicates, marking invalid values
as missing, and (for large data) a fixed stratified sample. Everything that is *fitted*
(imputation, scaling, TF-IDF, resampling) happens later inside each training fold.

Writes `data/processed/oilspill.parquet` and the fixed 5x2 outer folds `data/splits/oilspill_outer.npz`.
Both are committed, so every experiment notebook (and your teammate) uses identical data and folds.
Raw data is expected in `data/raw/` (not committed; see `data/README.md`).

In [1]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


Environment OK: scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, lightgbm 4.7.0


## Load
`oil_spill.csv`: 937 ocean-surface patches from satellite radar images, 49 numeric features.
The label column `class` holds `'1'` (oil slick) and `'-1'` (look-alike).

In [2]:
import numpy as np
import pandas as pd
from common import data, settings
from common.paths import RAW_DIR

raw = pd.read_csv(RAW_DIR / "oil_spill.csv")
print(raw.shape)
raw["class"].value_counts()

(937, 50)


class
'-1'    896
'1'      41
Name: count, dtype: int64

## Clean (stateless steps only)
* Label: strip quotes, `1` -> 1 (minority, oil slick), `-1` -> 0.
* Drop `attr1`: it is the patch number (1-352), an identifier rather than a measurement.
* Drop `attr23`: constant in every row.
* Remove duplicate rows (there are none, checked below).

Imputation and scaling are **not** done here; they are fitted inside each training fold.

In [3]:
df = raw.copy()
df["y"] = df.pop("class").astype(str).str.strip("'\" ").map({"1": 1, "-1": 0}).astype(int)
constant = [c for c in df.columns if c != "y" and df[c].nunique() == 1]
print("constant columns:", constant)
df = df.drop(columns=["attr1"] + constant)
print("duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
df["y"].value_counts()

constant columns: ['attr23']
duplicate rows: 0
(937, 48)


y
0    896
1     41
Name: count, dtype: int64

## Save processed data and the 5x2 outer folds

In [4]:
data.save_processed("oilspill", df)
data.make_outer_splits(df["y"], "oilspill")
data.summary("oilspill")

{'dataset': 'oilspill',
 'n': 937,
 'd': 47,
 'n_minority': 41,
 'minority_pct': 4.376,
 'IR': 21.85,
 'nan_cells': 0}